# Proyecto de Análisis númerico
## Uso de la FFT para procesamiento de imágenes

**Tema seleccionado:** detección y análisis de desenfoque en imágenes usando la Transformada Rápida de Fourier en dos dimensiones.

**Objetivo general:** aplicar la FFT 2D para transformar una imagen del dominio espacial al dominio de frecuencias, visualizar su espectro, estudiar cómo el desenfoque afecta las frecuencias altas y construir un detector básico de desenfoque.

**Preguntas guía del cuaderno:**

1. ¿Cómo se representa una imagen en el dominio de frecuencias?
2. ¿Qué ocurre con el espectro de Fourier cuando una imagen se suaviza o se desenfoca?
3. ¿Se puede usar la energía de frecuencias altas para clasificar una imagen como borrosa o no borrosa?
4. ¿Cómo afecta el parámetro de umbral a la clasificación?

## 1. Fundamento matemático mínimo

Una imagen en escala de grises puede modelarse como una matriz $f(x,y)$ de tamaño $M \times N$, donde cada entrada representa la intensidad de un píxel.

La Transformada Discreta de Fourier bidimensional se define como:

$$
F(u,v)=\sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)
e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

La transformada inversa permite regresar al dominio espacial:

$$
f(x,y)=\frac{1}{MN}\sum_{u=0}^{M-1}\sum_{v=0}^{N-1} F(u,v)
e^{j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

En imágenes:

- Las **bajas frecuencias** representan zonas suaves o cambios lentos de intensidad.
- Las **altas frecuencias** representan bordes, detalles finos, textura y ruido.
- Una imagen borrosa pierde detalles; por tanto, su energía en altas frecuencias disminuye.

## 2. Librerías necesarias

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy import ndimage
from numpy.fft import fft2, ifft2, fftshift, ifftshift

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["image.cmap"] = "gray"

## 3. Cargar o generar una imagen de prueba

Coloca una imagen propia dentro de una carpeta llamada `data` con el nombre `imagen.jpg`, `imagen.png` o cambia la ruta en la variable `ruta_imagen`.

Si no se encuentra una imagen externa, el cuaderno usa una imagen de ejemplo de `skimage`. Si `skimage` tampoco está instalado, genera una imagen sintética con patrones geométricos.

In [ ]:
def normalizar_imagen(img):
    # Convierte una imagen a flotante en el intervalo [0, 1].
    img = np.asarray(img, dtype=float)
    if img.ndim == 3:
        # Conversión simple RGB -> escala de grises
        img = 0.2989*img[..., 0] + 0.5870*img[..., 1] + 0.1140*img[..., 2]
    img = img - img.min()
    if img.max() > 0:
        img = img / img.max()
    return img

ruta_imagen = "data/Imagen_nitida.jpeg"

if os.path.exists(ruta_imagen):
    from PIL import Image
    imagen = normalizar_imagen(Image.open(ruta_imagen))
else:
    # Si no se encuentra la imagen, se intenta cargar una imagen de ejemplo de skimage. 
    try:
        from skimage import data
        imagen = normalizar_imagen(data.camera())
    except Exception:
        # Imagen sintética: cuadros, líneas y círculos para producir altas frecuencias
        n = 256
        y, x = np.ogrid[:n, :n]
        imagen = np.zeros((n, n), dtype=float)
        imagen[40:90, 40:210] = 1
        imagen[120:190, 30:100] = 0.7
        imagen[:, ::16] = 0.5
        imagen[::16, :] = 0.5
        mascara = (x - 180)**2 + (y - 160)**2 < 35**2
        imagen[mascara] = 1

plt.imshow(imagen)
plt.title("Imagen original en escala de grises")
plt.axis("off")
plt.show()

print("Dimensiones de la imagen:", imagen.shape)
print("Valor mínimo:", imagen.min(), "Valor máximo:", imagen.max())

## 4. Visualización de la FFT 2D

El resultado de la FFT es complejo. Para visualizarlo se usa el **espectro de magnitud**:

$$
S(u,v)=\log(1+|F(u,v)|)
$$

Se aplica `fftshift` para colocar las bajas frecuencias en el centro de la imagen, lo cual facilita la interpretación visual.

In [ ]:
def espectro_fft(imagen):
    """Devuelve FFT 2D, FFT centrada y espectro logarítmico normalizado."""
    F = fft2(imagen)
    F_centrada = fftshift(F)
    magnitud = np.log1p(np.abs(F_centrada))
    magnitud = normalizar_imagen(magnitud)
    return F, F_centrada, magnitud

F, F_centrada, magnitud = espectro_fft(imagen)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(imagen)
ax[0].set_title("Dominio espacial")
ax[0].axis("off")
ax[1].imshow(magnitud)
ax[1].set_title("Dominio de frecuencias: log(1 + |FFT|)")
ax[1].axis("off")
plt.show()

## 5. Experimento 1: efecto del desenfoque en el espectro

Se aplican diferentes niveles de desenfoque gaussiano. A mayor valor de $\sigma$, mayor suavizado de la imagen.

La hipótesis es:

> Al aumentar el desenfoque, disminuye la información de altas frecuencias.

In [ ]:
sigmas = [0, 1, 2, 4, 8]
imagenes_blur = []

for sigma in sigmas:
    if sigma == 0:
        img_blur = imagen.copy()
    else:
        img_blur = ndimage.gaussian_filter(imagen, sigma=sigma)
    imagenes_blur.append(img_blur)

fig, ax = plt.subplots(len(sigmas), 2, figsize=(10, 4*len(sigmas)))

for i, (sigma, img_blur) in enumerate(zip(sigmas, imagenes_blur)):
    _, _, mag = espectro_fft(img_blur)
    ax[i, 0].imshow(img_blur)
    ax[i, 0].set_title(f"Imagen con sigma = {sigma}")
    ax[i, 0].axis("off")
    ax[i, 1].imshow(mag)
    ax[i, 1].set_title(f"Espectro FFT con sigma = {sigma}")
    ax[i, 1].axis("off")

plt.tight_layout()
plt.show()

## 6. Métrica de energía en altas frecuencias

Para medir el detalle de la imagen se elimina una región central del espectro. Esa región central contiene principalmente bajas frecuencias. Luego se calcula el promedio de la magnitud restante.

Procedimiento:

1. Calcular $F(u,v)$ con `fft2`.
2. Centrar el espectro con `fftshift`.
3. Anular un cuadrado central de tamaño controlado por `radio`.
4. Regresar con `ifftshift`.
5. Aplicar `ifft2`.
6. Calcular la media de `20 log(|reconstrucción|)`.

Si la media es baja, se interpreta que la imagen tiene poca energía de alta frecuencia y probablemente está borrosa.

In [ ]:
def detectar_blur_fft(img, radio=30, umbral=10, visualizar=False):
    """
    Detector básico de desenfoque con FFT.

    Parámetros:
    - img: imagen en escala de grises normalizada.
    - radio: mitad del tamaño del cuadrado central que se anula.
    - umbral: valor límite para clasificar como borrosa.
    - visualizar: si True, muestra imagen y espectro.

    Retorna:
    - media: valor promedio de magnitudes después de eliminar bajas frecuencias.
    - borrosa: True si media <= umbral.
    """
    img = normalizar_imagen(img)
    h, w = img.shape
    cY, cX = h // 2, w // 2

    F = fft2(img)
    F_shift = fftshift(F)

    if visualizar:
        magnitud = np.log1p(np.abs(F_shift))
        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        ax[0].imshow(img)
        ax[0].set_title("Imagen de entrada")
        ax[0].axis("off")
        ax[1].imshow(normalizar_imagen(magnitud))
        ax[1].set_title("Espectro de magnitud")
        ax[1].axis("off")
        plt.show()

    # Remover bajas frecuencias del centro
    F_shift_filtrada = F_shift.copy()
    F_shift_filtrada[cY-radio:cY+radio, cX-radio:cX+radio] = 0

    # Reconstrucción usando principalmente altas frecuencias
    reconstruccion = ifft2(ifftshift(F_shift_filtrada))
    magnitud_reconstruida = 20 * np.log(np.abs(reconstruccion) + 1e-8)
    media = float(np.mean(magnitud_reconstruida))

    borrosa = media <= umbral
    return media, borrosa

media, borrosa = detectar_blur_fft(imagen, radio=30, umbral=10, visualizar=True)
print(f"Media FFT altas frecuencias: {media:.4f}")
print("Clasificación:", "Borrosa" if borrosa else "No borrosa")

## 7. Experimento 2: clasificación con distintos niveles de desenfoque

Aquí se evalúa la métrica anterior sobre imágenes con valores crecientes de $\sigma$. Se espera que la media disminuya conforme aumenta el desenfoque.

In [ ]:
radio = 30
umbral = 10
resultados = []

for sigma, img_blur in zip(sigmas, imagenes_blur):
    media, borrosa = detectar_blur_fft(img_blur, radio=radio, umbral=umbral)
    resultados.append({ 
        "sigma_blur": sigma,
        "media_altas_frecuencias": media,
        "umbral": umbral,
        "clasificacion": "Borrosa" if borrosa else "No borrosa"
    })

df_resultados = pd.DataFrame(resultados)
df_resultados

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df_resultados["sigma_blur"], df_resultados["media_altas_frecuencias"], marker="o")
plt.axhline(umbral, linestyle="--", label=f"Umbral = {umbral}")
plt.xlabel("Sigma del desenfoque gaussiano")
plt.ylabel("Media de altas frecuencias")
plt.title("Efecto del desenfoque sobre la energía de altas frecuencias")
plt.legend()
plt.grid(True)
plt.show()

## 8. Experimento 3: búsqueda de umbral

Como el método depende de un umbral, se prueban varios valores. Esta parte sirve para justificar en el informe por qué se escogió cierto umbral.